In [3]:
import math
import random

import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

import torch
from torch.autograd import Variable
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from tqdm.notebook import tqdm

from copy import deepcopy

import shutil
import sys
from importlib import reload

In [4]:
from google.colab import drive
from google.colab import files
drive.mount("/content/drive/Python/")

Mounted at /content/drive/


In [5]:
%cd /content/

/content


In [6]:
%pwd

'/content'

In [7]:
%ls

drive/  sample_data/


In [8]:
path = "/content/drive/My Drive/"
sys.path.insert(0, path)

In [9]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

In [10]:
device

device(type='cpu')

In [11]:
# За детерминизм!
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
torch.random.manual_seed(42)
torch.cuda.random.manual_seed_all(42)

# Arithmetic Dataset

In [12]:
class ArithmeticDataset(Dataset):
    def __init__(self, dataset_size=100, left_int=-2**15+1, right_int=2**15-1, len_code=32):
        super().__init__()
        #only integer type
        if left_int > right_int or len_code < 0 or not isinstance(len_code, int) :
          return -1
        self.dataset_size = dataset_size
        self.left_int = left_int
        self.right_int = right_int
        self.len_code = len_code
        self.points, self.targets = self._generate(dataset_size, left_int, right_int, len_code)
    
    @staticmethod
    def ten2two(number, len=32):
        res = np.zeros(len)
        ind_minus = 0

        if number < 0:
          res[0] = 1
          ind_minus = 1

        num = np.abs(deepcopy(number))

        for i in range(len-1, -1, -1):
          if num == 0:
            break
          res[i] = num % 2
          num = num // 2
        
        if ind_minus == 0:
          return res
        elif ind_minus == 1:
          acc = 1
          res_ = []
          for point in res[::-1][:(len-1)]:
            if point == 0:
              if acc == 1:
                res_.append(0)
                acc = 1
              elif acc == 0:
                res_.append(1)
                acc = 0
            elif point == 1:
              if acc == 1:
                res_.append(1)
                acc = 0
              elif acc == 0:
                res_.append(0)
                acc = 0

          res_.append(1)

          return np.array(res_[::-1])
        
    @staticmethod
    def two2ten(arr):
      sign = 0
      result = 0
      if arr[0] == 1:
        sign = 1
        acc = 1
        arr_ = []
        for point in arr[::-1]:
          if point == 0:
            if acc == 1:
              arr_.append(0)
              acc = 1
            elif acc == 0:
              arr_.append(1)
              acc = 0
          elif point == 1:
            if acc == 1:
              arr_.append(1)
              acc = 0
            elif acc == 0:
              arr_.append(0)
              acc = 0
        
        arr = np.array(arr_[::-1])

      for i in range(1, len(arr)):
        result += arr[i]*2**(len(arr)-i-1)

      if sign == 0:
        return result
      else:
        return (-1)*result

    @staticmethod
    def flatten(t):
      return np.array([item for sublist in t for item in sublist])

    def _generate(self, size, left, right, len_code):
        points = np.array([np.round((right-left)*np.random.random(size)+left), np.round((right-left)*np.random.random(size)+left), 4*np.random.random(size)]).T

        targets = []
        points_bpe = []
        for point in tqdm(points):
          if 0. <= point[2] <= 1.:
            targets.append(self.ten2two(point[0] + point[1], len_code))
            point_bpe = [self.ten2two(point[0], len_code), self.ten2two(point[1], len_code), np.array([1., 0., 0., 0.])]
            points_bpe.append(self.flatten(point_bpe))
          elif 1. <= point[2] <= 2.:
            targets.append(self.ten2two(point[0] - point[1], len_code))
            point_bpe = [self.ten2two(point[0], len_code), self.ten2two(point[1], len_code), np.array([0., 1., 0., 0.])]
            points_bpe.append(self.flatten(point_bpe))
          elif 2. <= point[2] <= 3.:
            targets.append(self.ten2two(point[0] * point[1], len_code))
            point_bpe = [self.ten2two(point[0], len_code), self.ten2two(point[1], len_code), np.array([0., 0., 1., 0.])]
            points_bpe.append(self.flatten(point_bpe))
          elif 3. <= point[2] <= 4.:
            if point[1] == 0:
              point[1] += 1
            targets.append(self.ten2two(point[0] // point[1], len_code))
            point_bpe = [self.ten2two(point[0], len_code), self.ten2two(point[1], len_code), np.array([0., 0., 0., 1.])]
            points_bpe.append(self.flatten(point_bpe)) 
        targets = np.array(targets)
        points_bpe = np.array(points_bpe)

        return points_bpe, targets

    def show(self, size=5):
        if size < self.dataset_size:
          signs = []
          for point in self.points[:size]:
            if point[2*self.len_code] > 0.5:
              sign = '+'
            elif point[2*self.len_code + 1] > 0.5:
              sign = '-'
            elif point[2*self.len_code + 2] > 0.5:
              sign = '*'
            elif point[2*self.len_code + 3] > 0.5:
              sign = '//'
            signs.append(sign)

          return (self.points[:size], self.targets[:size],
              *zip(list(map(lambda x: self.two2ten(x[0:self.len_code]), self.points[:size])),
              list(map(lambda x: self.two2ten(x[self.len_code:2*self.len_code]), self.points[:size]))),
              list(map(lambda x: self.two2ten(x), self.targets[:size])),
              signs)
        else:
          return -1

    def __len__(self):
        return self.dataset_size

    def __getitem__(self, idx):
        if torch.is_tensor(idx):
          idx = idx.tolist()      
        sample = (torch.tensor(self.points[idx], dtype=torch.double, device=device), torch.tensor(self.targets[idx], dtype=torch.double, device=device))

        return sample

In [13]:
ArithmeticDataset.ten2two(-456)

array([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 0, 0, 0, 1, 1, 1, 0, 0, 0])

In [14]:
ArithmeticDataset.two2ten([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 0, 0, 0, 1, 1, 1, 0, 0, 0])

-456

# Train function

In [15]:
def train(model, train_dataloader, valid_dataloader, epochs, criterion, optimizer, scheduler=None, clip=10, name='Calculus.pt'):

    loss_epochs_train = []
    loss_epochs_valid = []
    tolerance = 5
    cur_tolerance = 0
    best_epoch_valid_loss = np.inf

    for idx in range(epochs):
      loss_samples_train = []
      loss_samples_valid = []

      for sample in tqdm(train_dataloader):
        optimizer.zero_grad()   # zero the gradient buffers
        point, target = sample
        output = model(point)
        output = output.squeeze(1)
        # print(point, output, target)
        loss = criterion(output, target.to(device))
        loss.backward()
        loss_samples_train.append(loss.data.cpu().numpy())
        if clip is not None:
          nn.utils.clip_grad_norm_(model.parameters(), clip, error_if_nonfinite=False)
        optimizer.step()    # Does the update

      if scheduler is not None:
        scheduler.step()
      
      with torch.no_grad():
        for sample in tqdm(valid_dataloader):
          point, target = sample
          output = model(point)
          output = output.squeeze(1)
          # print(point, output, target)
          loss = criterion(output, target.to(device))
          loss_samples_valid.append(loss.data.cpu().numpy())

      loss_samples_train_mean = np.array(loss_samples_train).mean()
      loss_samples_valid_mean = np.array(loss_samples_valid).mean()
      print(f"Epoch {idx: > 8} Loss_train: {loss_samples_train_mean:8.2f} Loss_valid: {loss_samples_valid_mean:8.2f}")
      loss_epochs_train.append(loss_samples_train_mean)
      loss_epochs_valid.append(loss_samples_valid_mean)

      average_epoch_valid_loss = sum(loss_samples_valid) / len(loss_samples_valid)
      if best_epoch_valid_loss >= average_epoch_valid_loss:
          torch.save(model.state_dict(), name)
          shutil.copy(name, path + name)
          print('Best model saved!')
          cur_tolerance = 0
          best_epoch_valid_loss = average_epoch_valid_loss
      else:
          cur_tolerance += 1
          if cur_tolerance >= tolerance:
            print('Overteaching! Best model has been saved!')
            break;

    plt.plot(loss_epochs_train, label='train')
    plt.plot(loss_epochs_valid, label='valid')
    plt.ylabel('Loss')
    plt.xlabel('Epoch')
    plt.legend()
    plt.show()

    return (loss_epochs_train, loss_epochs_valid)

# Test function

In [16]:
def test(network, test_dataloader, cast_func=None):

    differences_all = None
    points = None
    targets = None
    outputs = None
    with torch.no_grad():
      for sample in tqdm(test_dataloader):
        point, target = sample
        output = network(point)
        output = output.squeeze(1)
        if cast_func is not None:
          output = cast_func(output)
        # print(point, output, target)
        result = torch.tensor((output > 0.5*torch.ones_like(output)), dtype=torch.float64)
        differences = torch.tensor((result == target), dtype=torch.float64).sum(dim=1)
        if differences_all == None:
          differences_all = differences.clone().detach()
          points = point.clone().detach()
          targets = target.clone().detach()
          outputs = output.clone().detach()
        else:
          differences_all = torch.cat((differences_all, differences))
          points = torch.cat((points, point))
          targets = torch.cat((targets, target))
          outputs = torch.cat((outputs, output))

    print(differences_all.mean().item(), '\n')
    return points, targets, outputs

## Dataset

In [15]:
DATASET_SIZE = 5000000
arith_train = ArithmeticDataset(DATASET_SIZE)
arith_valid = ArithmeticDataset(DATASET_SIZE//1000)
arith_test = ArithmeticDataset(DATASET_SIZE//1000)

  0%|          | 0/5000000 [00:00<?, ?it/s]

  0%|          | 0/5000 [00:00<?, ?it/s]

  0%|          | 0/5000 [00:00<?, ?it/s]

In [37]:
arith_train.show(3)

(array([[1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
         1., 1., 0., 1., 1., 1., 1., 1., 1., 1., 1., 0., 0., 0., 1., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 1., 0., 0., 0., 1., 0., 0., 0., 0., 1., 1.,
         0., 0., 0., 1.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 1., 1., 1., 0., 0., 1., 1., 0., 1., 1., 0., 0., 0., 0., 1.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 1., 1., 0., 1., 1., 1., 1., 1., 1., 1., 1., 0., 1., 0., 0.,
         0., 1., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 1., 1., 1., 0., 1., 1., 0., 1., 1., 0., 0., 0., 1., 1.,
         1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
         1., 1., 1., 1., 0., 0., 1., 1., 0., 1., 1., 1., 0., 0., 0., 1.,
         0., 0., 0., 1.]]),
 array([[1., 1., 1., 1., 1., 1., 1., 1., 1.,

In [38]:
arith_train[0:3]

(tensor([[1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
          0., 1., 1., 1., 1., 1., 1., 1., 1., 0., 0., 0., 1., 0., 0., 0., 0., 0.,
          0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1.,
          0., 0., 0., 1., 0., 0., 0., 0., 1., 1., 0., 0., 0., 1.],
         [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1.,
          1., 1., 0., 0., 1., 1., 0., 1., 1., 0., 0., 0., 0., 1., 0., 0., 0., 0.,
          0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 1., 0., 1., 1.,
          1., 1., 1., 1., 1., 1., 0., 1., 0., 0., 0., 1., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          1., 1., 1., 0., 1., 1., 0., 1., 1., 0., 0., 0., 1., 1., 1., 1., 1., 1.,
          1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 0., 0.,
          1., 1., 0., 1., 1., 1., 0., 0., 0., 1., 0., 0., 0., 1.]],
        device='cuda:0', dtype=torch.float64),
 tensor([[1.,

In [39]:
len(arith_train)

5000000

In [40]:
data_train = DataLoader(arith_train, batch_size=10000)
data_valid = DataLoader(arith_valid, batch_size=10000)
data_test = DataLoader(arith_test, batch_size=10000)

In [41]:
next(iter(data_train))

[tensor([[1., 1., 1.,  ..., 0., 0., 1.],
         [0., 0., 0.,  ..., 1., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 1.],
         ...,
         [0., 0., 0.,  ..., 1., 0., 0.],
         [1., 1., 1.,  ..., 0., 0., 0.],
         [1., 1., 1.,  ..., 0., 1., 0.]], device='cuda:0', dtype=torch.float64),
 tensor([[1., 1., 1.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 1., 0., 1.],
         [1., 1., 1.,  ..., 0., 1., 1.],
         ...,
         [0., 0., 0.,  ..., 0., 1., 0.],
         [1., 1., 1.,  ..., 0., 1., 0.],
         [0., 0., 0.,  ..., 1., 1., 0.]], device='cuda:0', dtype=torch.float64)]

## Net

In [20]:
class Net(nn.Module):

    def __init__(self, size=(68, 150, 32), hidden=5):
        super(Net, self).__init__()
        # an affine operation: y = Wx + b
        self.size = size
        self.hidden = hidden
        self.fc1 = nn.Linear(size[0], size[1])
        layers = []
        for i in range(hidden):
          layers.append(nn.Linear(size[1], size[1]))
        self.fc = nn.ModuleList(layers)
        self.fc3 = nn.Linear(size[1], size[2])

    def forward(self, x):
        # x = nn.Tanhshrink()(self.fc1(x))
        x = nn.Sigmoid()(self.fc1(x))
        for i in range(self.hidden):
          x = nn.Sigmoid()(self.fc[i](x))
        x = self.fc3(x)
        return x

net = Net().double().to(device)
print(net)

Net(
  (fc1): Linear(in_features=68, out_features=150, bias=True)
  (fc): ModuleList(
    (0): Linear(in_features=150, out_features=150, bias=True)
    (1): Linear(in_features=150, out_features=150, bias=True)
    (2): Linear(in_features=150, out_features=150, bias=True)
    (3): Linear(in_features=150, out_features=150, bias=True)
    (4): Linear(in_features=150, out_features=150, bias=True)
  )
  (fc3): Linear(in_features=150, out_features=32, bias=True)
)


## Training

In [ ]:
EPOCHS_TO_TRAIN = 50
LR = 1e-2
optimizer = optim.Adam(net.parameters(), lr=LR)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.7)
pos_weight = torch.tensor([2 if i==0 else 1 for i in range(32)], dtype=torch.float64, device=device)

train(model=net,
      train_dataloader=data_train,
      valid_dataloader=data_valid,
      epochs=EPOCHS_TO_TRAIN,
      criterion=nn.BCEWithLogitsLoss(pos_weight=pos_weight),  #nn.MSELoss() #nn.BCELoss() 
      optimizer=optimizer,
      scheduler=scheduler,
      clip=10)

  0%|          | 0/500 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

Epoch        0 Loss_train:     0.64 Loss_valid:     0.58
Best model saved!


  0%|          | 0/500 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

...

Epoch       32 Loss_train:     0.26 Loss_valid:     0.27
Best model saved!


  0%|          | 0/500 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

Epoch       33 Loss_train:     0.26 Loss_valid:     0.27
Best model saved!


  0%|          | 0/500 [00:00<?, ?it/s]

## Testing

In [22]:
net.load_state_dict(torch.load(path + 'Calculus.pt', map_location=device))

<All keys matched successfully>

In [23]:
result = test(network=net, test_dataloader=data_test, cast_func=nn.Sigmoid())

  0%|          | 0/1 [00:00<?, ?it/s]

26.355 



/usr/local/lib/python3.7/dist-packages/ipykernel_launcher.py:15: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  from ipykernel import kernelapp as app
/usr/local/lib/python3.7/dist-packages/ipykernel_launcher.py:16: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  app.launch_new_instance()


## Final precision:

In [24]:
26.355/32*100

82.359375

In [26]:
list(zip(*result))[:3]

[(tensor([1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
          0., 1., 1., 1., 1., 1., 1., 1., 1., 0., 0., 0., 1., 0., 1., 1., 1., 1.,
          1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 0., 0., 1.,
          0., 0., 1., 1., 0., 0., 0., 1., 1., 0., 0., 1., 0., 0.],
         dtype=torch.float64),
  tensor([1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
          1., 1., 1., 0., 1., 1., 0., 0., 0., 1., 1., 1., 0., 0.],
         dtype=torch.float64),
  tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
          1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
          1.0000, 1.0000, 0.9996, 0.1569, 0.4322, 0.4898, 0.4874, 0.4743, 0.4912,
          0.4978, 0.5011, 0.4937, 0.5393, 0.5308], dtype=torch.float64)),
 (tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1.,
          1., 1., 0., 0., 1., 1., 0., 1., 1., 0., 0., 0., 0., 1., 1., 1., 